# LFM2-ColBERT Actual Zero-Shot Scripts Notebook

Generated: **2026-02-27 09:03:49Z**

This notebook gives the **actual runnable zero-shot workflow** using project scripts.
It is intended for reproducible execution and sharing.


## What This Notebook Runs

1. Build code-switch baseline dataset
2. FR/WO/ENG zero-shot run
3. Waxal zero-shot-only run (`plain` + `codeswitch`)
4. Bargain scenario zero-shot run
5. Consolidate metrics into a summary table


In [1]:
from pathlib import Path
import subprocess
import json
import pandas as pd

REPO = Path(r'C:\Users\momo-\OneDrive\Desktop\YAATAL\Yaatal-Engine')
PYTHON = REPO / '.venv-colbert' / 'Scripts' / 'python.exe'

assert REPO.exists(), REPO
assert PYTHON.exists(), PYTHON
print('repo:', REPO)
print('python:', PYTHON)


repo: C:\Users\momo-\OneDrive\Desktop\YAATAL\Yaatal-Engine
python: C:\Users\momo-\OneDrive\Desktop\YAATAL\Yaatal-Engine\.venv-colbert\Scripts\python.exe


In [2]:
# Actual scripts used in this study
SCRIPTS = [
    REPO / 'scripts' / 'build_codeswitch_baseline_v2.py',
    REPO / 'scripts' / 'run_lfm_colbert_fr_en_wo_iterations.py',
    REPO / 'scripts' / 'run_lfm_colbert_waxal.py',
    REPO / 'scripts' / 'run_lfm_colbert_scenario_bargain.py',
]
for p in SCRIPTS:
    print(p.name, 'exists=', p.exists())


build_codeswitch_baseline_v2.py exists= True
run_lfm_colbert_fr_en_wo_iterations.py exists= True
run_lfm_colbert_waxal.py exists= True
run_lfm_colbert_scenario_bargain.py exists= True


In [3]:
# Optional: print full source of a script (actual code)
def show_script(name: str):
    p = REPO / 'scripts' / name
    print('---', p, '---')
    print(p.read_text(encoding='utf-8'))

# Example:
show_script('run_lfm_colbert_waxal.py')


--- C:\Users\momo-\OneDrive\Desktop\YAATAL\Yaatal-Engine\scripts\run_lfm_colbert_waxal.py ---
#!/usr/bin/env python3
"""
Run zero-shot and fine-tune experiments for LiquidAI LFM2 ColBERT on WaxalNLP.

This script:
1) Streams a small ASR subset from google/WaxalNLP.
2) Builds a retrieval benchmark dataset (query -> relevant doc IDs).
3) Runs zero-shot retrieval evaluation.
4) Optionally fine-tunes the ColBERT model with contrastive training.
5) Writes metrics/artifacts for zero-shot-only or fine-tune flows.
"""

from __future__ import annotations

import argparse
import json
import math
import random
import re
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import trackio
from datasets import load_dataset
from pylate import indexes, losses, models, retrieve
from sentence_transformers import InputExample
from torch.utils.data import DataLoader


def _utc_now() -> str:
    return date

In [4]:
# 1) Build dataset (FR/WO/ENG + minimal FUL/YO)
cmd = [
    str(PYTHON),
    'scripts/build_codeswitch_baseline_v2.py',
    '--fr-wo-eng-docs', '120',
    '--ful-yo-docs', '40',
    '--output-dir', 'data/corpus/codeswitch_baseline_v2',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)


C:\Users\momo-\OneDrive\Desktop\YAATAL\Yaatal-Engine\.venv-colbert\Scripts\python.exe scripts/build_codeswitch_baseline_v2.py --fr-wo-eng-docs 120 --ful-yo-docs 40 --output-dir data/corpus/codeswitch_baseline_v2


CompletedProcess(args=['C:\\Users\\momo-\\OneDrive\\Desktop\\YAATAL\\Yaatal-Engine\\.venv-colbert\\Scripts\\python.exe', 'scripts/build_codeswitch_baseline_v2.py', '--fr-wo-eng-docs', '120', '--ful-yo-docs', '40', '--output-dir', 'data/corpus/codeswitch_baseline_v2'], returncode=0)

In [5]:
# 2) FR/WO/ENG zero-shot
cmd = [
    str(PYTHON),
    'scripts/run_lfm_colbert_fr_en_wo_iterations.py',
    '--model-id', 'LiquidAI/LFM2-ColBERT-350M',
    '--sample-size', '80',
    '--eval-queries', '40',
    '--encode-batch-size', '4',
    '--top-k', '10',
    '--output-root', r'C:\Users\momo-\OneDrive\Desktop\YAATAL\artifacts\lfm_colbert_fr_en_wo',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)


C:\Users\momo-\OneDrive\Desktop\YAATAL\Yaatal-Engine\.venv-colbert\Scripts\python.exe scripts/run_lfm_colbert_fr_en_wo_iterations.py --model-id LiquidAI/LFM2-ColBERT-350M --sample-size 80 --eval-queries 40 --encode-batch-size 4 --top-k 10 --output-root C:\Users\momo-\OneDrive\Desktop\YAATAL\artifacts\lfm_colbert_fr_en_wo


CompletedProcess(args=['C:\\Users\\momo-\\OneDrive\\Desktop\\YAATAL\\Yaatal-Engine\\.venv-colbert\\Scripts\\python.exe', 'scripts/run_lfm_colbert_fr_en_wo_iterations.py', '--model-id', 'LiquidAI/LFM2-ColBERT-350M', '--sample-size', '80', '--eval-queries', '40', '--encode-batch-size', '4', '--top-k', '10', '--output-root', 'C:\\Users\\momo-\\OneDrive\\Desktop\\YAATAL\\artifacts\\lfm_colbert_fr_en_wo'], returncode=0)

In [6]:
# 3) Waxal zero-shot-only (plain + codeswitch)
cmd = [
    str(PYTHON),
    'scripts/run_lfm_colbert_waxal.py',
    '--model-id', 'LiquidAI/LFM2-ColBERT-350M',
    '--eval-style', 'both',
    '--zero-shot-only',
    '--train-docs', '80',
    '--eval-docs', '60',
    '--eval-queries', '40',
    '--encode-batch-size', '4',
    '--top-k', '10',
    '--output-root', r'C:\Users\momo-\OneDrive\Desktop\YAATAL\artifacts\lfm_colbert_waxal',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)


C:\Users\momo-\OneDrive\Desktop\YAATAL\Yaatal-Engine\.venv-colbert\Scripts\python.exe scripts/run_lfm_colbert_waxal.py --model-id LiquidAI/LFM2-ColBERT-350M --eval-style both --zero-shot-only --train-docs 80 --eval-docs 60 --eval-queries 40 --encode-batch-size 4 --top-k 10 --output-root C:\Users\momo-\OneDrive\Desktop\YAATAL\artifacts\lfm_colbert_waxal


CompletedProcess(args=['C:\\Users\\momo-\\OneDrive\\Desktop\\YAATAL\\Yaatal-Engine\\.venv-colbert\\Scripts\\python.exe', 'scripts/run_lfm_colbert_waxal.py', '--model-id', 'LiquidAI/LFM2-ColBERT-350M', '--eval-style', 'both', '--zero-shot-only', '--train-docs', '80', '--eval-docs', '60', '--eval-queries', '40', '--encode-batch-size', '4', '--top-k', '10', '--output-root', 'C:\\Users\\momo-\\OneDrive\\Desktop\\YAATAL\\artifacts\\lfm_colbert_waxal'], returncode=0)

In [7]:
# 4) Bargain scenario zero-shot
cmd = [
    str(PYTHON),
    'scripts/run_lfm_colbert_scenario_bargain.py',
    '--pairs-jsonl', 'data/corpus/codeswitch_baseline_v2/pairs.jsonl',
    '--model-id', 'LiquidAI/LFM2-ColBERT-350M',
    '--max-docs', '120',
    '--eval-queries', '80',
    '--top-k', '10',
    '--encode-batch-size', '4',
    '--output-root', r'C:\Users\momo-\OneDrive\Desktop\YAATAL\artifacts\lfm_colbert_scenarios\bargain',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)


C:\Users\momo-\OneDrive\Desktop\YAATAL\Yaatal-Engine\.venv-colbert\Scripts\python.exe scripts/run_lfm_colbert_scenario_bargain.py --pairs-jsonl data/corpus/codeswitch_baseline_v2/pairs.jsonl --model-id LiquidAI/LFM2-ColBERT-350M --max-docs 120 --eval-queries 80 --top-k 10 --encode-batch-size 4 --output-root C:\Users\momo-\OneDrive\Desktop\YAATAL\artifacts\lfm_colbert_scenarios\bargain


CompletedProcess(args=['C:\\Users\\momo-\\OneDrive\\Desktop\\YAATAL\\Yaatal-Engine\\.venv-colbert\\Scripts\\python.exe', 'scripts/run_lfm_colbert_scenario_bargain.py', '--pairs-jsonl', 'data/corpus/codeswitch_baseline_v2/pairs.jsonl', '--model-id', 'LiquidAI/LFM2-ColBERT-350M', '--max-docs', '120', '--eval-queries', '80', '--top-k', '10', '--encode-batch-size', '4', '--output-root', 'C:\\Users\\momo-\\OneDrive\\Desktop\\YAATAL\\artifacts\\lfm_colbert_scenarios\\bargain'], returncode=0)

In [8]:
# 5) Load known run outputs and present clear table
fr_path = Path(r'C:\Users\momo-\OneDrive\Desktop\YAATAL\artifacts\lfm_colbert_fr_en_wo\run-20260227-083651\metrics.json')
wx_path = Path(r'C:\Users\momo-\OneDrive\Desktop\YAATAL\artifacts\lfm_colbert_waxal\run-20260227-083844\metrics.json')
sc_path = Path(r'C:\Users\momo-\OneDrive\Desktop\YAATAL\artifacts\lfm_colbert_scenarios\bargain\run-20260227-085047\metrics.json')

fr = json.loads(fr_path.read_text(encoding='utf-8'))
wx = json.loads(wx_path.read_text(encoding='utf-8'))
sc = json.loads(sc_path.read_text(encoding='utf-8'))

rows = []
for style, m in fr['metrics_by_style'].items():
    rows.append({'track': 'fr_wo_eng', 'style': style, 'MRR@10': m['mrr_at_k'], 'Recall@10': m['recall_at_k'], 'nDCG@10': m['ndcg_at_k'], 'queries': m['evaluated_queries']})
for style, m in wx['baseline_by_eval_style'].items():
    rows.append({'track': 'waxal_zeroshot', 'style': style, 'MRR@10': m['mrr_at_k'], 'Recall@10': m['recall_at_k'], 'nDCG@10': m['ndcg_at_k'], 'queries': m['evaluated_queries']})
for style, m in sc['metrics_by_style'].items():
    rows.append({'track': 'bargain_scenario', 'style': style, 'MRR@10': m['mrr_at_k'], 'Recall@10': m['recall_at_k'], 'nDCG@10': m['ndcg_at_k'], 'queries': m['evaluated_queries']})

df = pd.DataFrame(rows).sort_values(['track', 'style']).reset_index(drop=True)
display(df)


,track,style,MRR@10,Recall@10,nDCG@10,queries
0,bargain_scenario,codeswitch,1.000000,1.000,1.000000,80
1,bargain_scenario,plain,1.000000,1.000,1.000000,80
2,fr_wo_eng,english,0.229167,0.300,0.246212,40
3,fr_wo_eng,fr_en_wo_mix,1.000000,1.000,1.000000,40
4,fr_wo_eng,french,0.210238,0.300,0.231333,40
5,fr_wo_eng,wolof,1.000000,1.000,1.000000,40
6,waxal_zeroshot,codeswitch,0.504861,0.650,0.539183,40
7,waxal_zeroshot,plain,0.501528,0.575,0.518737,40


## Notes

- If you want fresh runs, uncomment the `subprocess.run(...)` lines and execute cells in order.
- This notebook is script-first: it references and runs the real project scripts directly.
